# Fine-Tune Gemma 4 for Alzheimer's Care
## Using Unsloth on Google Colab (FREE GPU)

This notebook fine-tunes Gemma 4 on medical Alzheimer's scenarios.

**Setup**: Just click "Run" on each cell in order. No configuration needed!

⏱️ **Estimated time**: 4-8 hours

## Step 1: Install Dependencies
This installs Unsloth and all required packages.

In [ ]:
!pip install "unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git"
!pip install torch torchvision torchaudio
!pip install transformers datasets peft trl accelerate bitsandbytes

'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.


## Step 2: Import Libraries

In [2]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
import torch
from datetime import datetime
import json

print("✅ All libraries imported successfully!")
print(f"🔥 GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

## Step 3: Upload Training Data

Upload your `alzheimers_training.jsonl` file from your computer.
If you don't have it, we'll create it automatically.

In [ ]:
import os

# Create training data if it doesn't exist
training_data = '''{
"prompt": "Patient is confused about the date and time. They keep asking 'What year is it?' repeatedly. What is your assessment?",
"completion": "This patient is showing signs of temporal disorientation, a key indicator of cognitive decline in Alzheimer's disease. Early-stage dementia typically presents with disorientation to time first, followed by place and person. Recommend MMSE scoring, neuropsychological evaluation, and MRI to rule out other causes. Consider referral to neurology for further assessment. Monitor for progression."
}
{
"prompt": "81-year-old patient forgot their afternoon medication again this week. They took it twice yesterday instead. How should we address medication adherence?",
"completion": "Medication non-adherence is critical in Alzheimer's. Recommend: (1) Simplify regimen if possible, (2) Use pill organizer with labeled days/times, (3) Set phone alarms or caregiver reminders, (4) Consider automated dispensers, (5) Weekly caregiver check-ins. For missed doses, advise NOT doubling doses. Document in medical record and discuss with family. Consider Donepezil 10mg daily if not already prescribed."
}
{
"prompt": "Patient left home at 2 AM and walked 3 miles away. Found by police. This is the third incident in two weeks. What emergency interventions are needed?",
"completion": "Wandering behavior indicates moderate-to-advanced cognitive decline and safety risk. Immediate actions: (1) Assess for underlying causes (pain, infection, medication side effects), (2) Implement 24-hour supervision or facility placement, (3) Medical alert bracelet with ID, (4) Notify local police non-emergency, (5) GPS tracking device, (6) Environmental modifications (alarm on doors), (7) Evaluate for agitation medication if appropriate. Discuss placement options with family."
}
{
"prompt": "Caregiver reports patient becoming aggressive when bathed or dressed. Hitting, yelling, refusing care. MMSE 14/30. What is happening and how to manage?",
"completion": "Behavioral aggression in moderate Alzheimer's (MMSE 10-20) often stems from: (1) Loss of autonomy/dignity perception, (2) Pain or discomfort, (3) Overstimulation, (4) Medication effects. Management: (1) Validate emotions without arguing, (2) Use simple commands one at a time, (3) Maintain routine, (4) Distract with music/activity, (5) Avoid confrontation, (6) Evaluate for pain/UTI/infection, (7) Consider low-dose antipsychotic if severe (risperidone 0.5mg BID), (8) Caregiver respite care."
}
{
"prompt": "Patient scored MMSE 22/30 last month, now 18/30 this month. Rapid decline over 4 weeks. What does this indicate?",
"completion": "Rapid cognitive decline of 4 points in 4 weeks is abnormal and concerning. Differential diagnosis: (1) Acute delirium from infection/UTI, (2) Medication toxicity (check recent additions), (3) Hypothyroidism, (4) B12 deficiency, (5) Depression, (6) Rapid Alzheimer's progression. URGENT: Order CBC, metabolic panel, TSH, B12, urinalysis. Neuroimaging if first presentation. Rule out treatable causes before attributing to Alzheimer's progression. Document in chart with intervention plan."
}
{
"prompt": "Patient keeps calling for deceased spouse 'Where is Margaret? Why hasn't Margaret come to pick me up?' How should staff respond?",
"completion": "Responding to false memories/deceased person requests in Alzheimer's requires validation therapy: (1) DO NOT correct or argue about reality, (2) Acknowledge emotions ('You miss Margaret'), (3) Redirect to pleasant activities, (4) Use life review/reminiscence therapy, (5) Play familiar music, (6) Involve in structured activities. This is normal in mid-to-late stage Alzheimer's. Document frequency. Consider therapeutic response training for caregivers."
}
{
"prompt": "Family concerned about fall risk. Patient shuffles when walking, stiff movements, occasional tremor in hands. On Donepezil and Memantine. Could medications be causing this?",
"completion": "Gait abnormalities and tremor in Alzheimer's can be: (1) Disease progression (Lewy body dementia features), (2) Parkinson's-related, (3) Medication effects less likely with Donepezil/Memantine (not typical culprits). Assessment: (1) Orthostatic vital signs, (2) Check for medication interactions, (3) PT/OT evaluation, (4) Environmental fall risk assessment, (5) Consider assistive device (walker, handrails), (6) Physical therapy 2-3x/week. If tremor worsens, consider neurology referral for Parkinson's overlap."
}
{
"prompt": "Patient hasn't eaten in two days. Refusing food, saying 'I'm not hungry' and 'Food tastes like metal.' What could cause this?",
"completion": "Appetite loss and taste abnormalities in Alzheimer's can indicate: (1) Medication side effects (check recent additions), (2) Zinc/vitamin deficiency, (3) Dental problems/denture fit issues, (4) Mouth sores/infection, (5) Swallowing difficulty (dysphagia screening needed), (6) Depression, (7) Disease progression. Action: (1) Examine mouth/teeth, (2) Review medications, (3) Consider zinc/B-vitamin supplementation, (4) Offer favorite foods and textures, (5) Swallow study if indicated, (6) Occupational therapy for feeding adaptation. Monitor weight and nutritional status closely."
}
{
"prompt": "Patient with MMSE 8/30 incontinent during day and night. Family wants to know if this is normal progression.",
"completion": "Urinary incontinence in late-stage Alzheimer's (MMSE <10) is expected and reflects loss of voluntary bladder control. NOT treatable with medications. Management: (1) Schedule toileting every 2-3 hours, (2) Use incontinence products/briefs, (3) Monitor for UTI (fever, delirium), (4) Limit fluids 2 hours before bed, (5) Ensure accessible bathroom, (6) Skin care to prevent breakdown, (7) AVOID catheters unless absolutely necessary. Educate family this is disease progression, not something to fight. Focus on comfort and dignity."
}
{
"prompt": "Patient takes Memantine 20mg daily. Family concerned about side effects. Patient seems drowsy and confused. Is Memantine causing this?",
"completion": "NMDA antagonist effects (Memantine) typically include: dizziness, confusion, headache (early), but tolerance develops. However, over-sedation suggests: (1) Dosing too rapid or too high, (2) Drug interaction, (3) Other medications (benzodiazepines? check), (4) Underlying delirium. Action: (1) Review drug interactions, (2) Check timing of dosing, (3) Verify dose is 20mg (should be titrated slowly), (4) Evaluate for infection/delirium, (5) Trial reducing to 10mg if recently increased, (6) Monitor 1-2 weeks. Confusion may also be disease progression. Document and discuss with prescriber."
}
{
"prompt": "Caregiver reports patient is having 'sundowning' - getting increasingly agitated, confused, and distressed in late afternoon/evening. Any way to manage this?",
"completion": "Sundowning is common in Alzheimer's evening hours and relates to circadian rhythm disruption, fatigue, and reduced environmental cues. Management strategies: (1) Increase activity/sunlight exposure in morning-afternoon, (2) Structured routine with consistent mealtimes, (3) Reduce stimulation late afternoon (dim lights, quiet), (4) Simple dinner early, (5) Therapeutic music or activities, (6) Avoid caffeine, (7) Evening walk if safe, (8) Establish calming bedtime routine. Medication: LOW-dose melatonin 3-5mg or trazodone 25-50mg if severe, discuss with MD. Validate emotions, avoid confrontation."
}
{
"prompt": "Patient MMSE 16/30 on Donepezil 10mg for 18 months. Showing no improvement, some decline. Is continuing medication justified?",
"completion": "Donepezil slows cognitive decline but doesn't reverse it. After 18 months on 10mg with evidence of decline (baseline unknown), options: (1) Continue if maintaining function, (2) Add Memantine 20mg if not already on it (synergistic effect), (3) Review for treatable causes of decline (anemia, thyroid, B12), (4) Ensure medication adherence, (5) Neuropsych testing to objectively track trajectory. Decision to continue depends on patient/family goals. Document reasoning. Some families prefer continuation for perceived benefit; others stop due to cost/burden. Shared decision-making critical."
}
{
"prompt": "During visit, patient with Alzheimer's repeatedly tells the same story about their childhood for 30 minutes. Is this a sign of progression?",
"completion": "Repetitive storytelling/reminiscence is typical in Alzheimer's, especially mid-stage (MMSE 10-20). It reflects: (1) Short-term memory loss but preserved remote memory, (2) Loss of awareness they've told story before, (3) Seeking connection/validation. This is NORMAL, not necessarily progression. Management: (1) Listen patiently and validate, (2) Gently redirect when appropriate, (3) Use life review therapy, (4) Avoid correcting or showing frustration, (5) Engage family in reminiscence activities. Monitor for increase in repetition or new concerning behaviors. Document frequency. Reassure family this is expected."
}
'''

# Save training data
training_file = '/content/alzheimers_training.jsonl'
with open(training_file, 'w') as f:
    f.write(training_data.strip())

print(f"✅ Training data ready: {training_file}")
print(f"📊 Training examples: 12")

## Step 4: Load Gemma 4 Model
Downloads and initializes the model with LoRA adapters for efficient fine-tuning.

In [ ]:
print("[1/3] Loading Gemma 4 with Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-9b-bnb-4bit",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

print("✅ Model loaded!")
print(f"\n[2/3] Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    use_rslora=True,
)

print("✅ LoRA adapters configured!")

## Step 5: Set Up Training
Configure training parameters (learning rate, batch size, number of epochs).

In [ ]:
print("[3/3] Setting up trainer...")

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=3,  # 3 passes through data
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    output_dir="outputs",
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=training_file,
    dataset_text_field="completion",
    max_seq_length=2048,
)

print("✅ Trainer ready!")

## Step 6: Start Fine-Tuning 🚀
This is where the magic happens! Training will take 2-4 hours on Colab GPU.

You can close this tab and come back later. Training continues in background.

In [ ]:
print("="*60)
print("🔥 STARTING FINE-TUNING 🔥")
print(f"⏱️  Start time: {datetime.now()}")
print("="*60)

trainer.train()

print("="*60)
print("✅ FINE-TUNING COMPLETE!")
print(f"⏱️  End time: {datetime.now()}")
print("="*60)

## Step 7: Save the Fine-Tuned Model

In [ ]:
import os
from datetime import datetime

output_dir = f"/content/gemma4-medical-ft-{datetime.now().strftime('%Y%m%d')}"
os.makedirs(output_dir, exist_ok=True)

print(f"Saving to: {output_dir}")

# Save model
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Model saved locally in Colab!")
print(f"\n📦 Output directory: {output_dir}")

## Step 8: Download Model to Your Computer
This prepares files for download.

In [ ]:
import shutil
from google.colab import files

# Create a zip file for easy download
zip_path = '/content/gemma4-medical-ft.zip'
shutil.make_archive('/content/gemma4-medical-ft', 'zip', output_dir)

print(f"📦 Created zip file: {zip_path}")
print("\n💾 DOWNLOAD: Click the download button that appears below")
print("\nAfter download, extract to: c:\\yourOwn\\backend\\models\\gemma4-medical-ft")
print("\nThen run: ollama create yourown-medical-gemma4 --from ./models/gemma4-medical-ft")

files.download(zip_path)

## ✅ Done!

### Next Steps on Your Computer:

1. **Extract the downloaded zip** to `c:\yourOwn\backend\models\gemma4-medical-ft`

2. **Deploy to Ollama**:
```bash
cd c:\yourOwn\backend
ollama create yourown-medical-gemma4 --from ./models/gemma4-medical-ft
```

3. **Update `.env`**:
```
OLLAMA_MODEL=yourown-medical-gemma4
```

4. **Restart your backend**:
```bash
npm start
```

Your app now has a **fine-tuned medical AI** specialized for Alzheimer's care! 🏥